# Campus SVI Availability — Colab runner

Grid-based assessment of street-view imagery **availability** across Indonesian university
campuses, comparing crowdsourced (Mapillary) against proprietary (Google) coverage.

**Metadata only** — no imagery is downloaded at any stage.

Boundaries are read from Google Drive and every result is written back to Drive, so an
interrupted run survives a runtime disconnect. Run the cells top to bottom, **one campus at a
time**: every fetch stage is resumable, so if the runtime drops, re-run the same cell and it
picks up where it stopped.

---
## 0. Setup


In [ ]:
#@title Install dependencies (~1-2 min)
!pip install -q geopandas pyogrio streetlevel aiohttp nest-asyncio requests

import nest_asyncio; nest_asyncio.apply()   # lets async run inside Colab
import streetlevel, aiohttp
print('streetlevel', getattr(streetlevel, '__version__', '?'), '| aiohttp', aiohttp.__version__)


In [ ]:
#@title Clone the repo
REPO_URL = 'https://github.com/aditpradana36/campus-svi-availability.git'  #@param {type:'string'}

import os, sys
REPO_DIR = '/content/campus-svi-availability'
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('repo:', os.getcwd())


### Mount Drive and point the pipeline at it

`BOUNDARY_DIR` is where your `ui.gpkg` / `itb.shp` files already live.
`OUTPUT_ROOT` is where `data/` and `outputs/` will be created.

They are separate settings, so boundaries can sit in a read-only shared folder while results
accumulate somewhere else. Pointing both at the same folder is fine too.


In [ ]:
#@title Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

BOUNDARY_DIR = '/content/drive/MyDrive/campus_svi/boundaries'  #@param {type:'string'}
OUTPUT_ROOT  = '/content/drive/MyDrive/campus_svi'             #@param {type:'string'}

from campus_svi import config
config.use_drive(boundary_dir=BOUNDARY_DIR, output_root=OUTPUT_ROOT)
for k, v in config.paths().items():
    print(f'{k:<16} {v}')

import os
if not os.path.isdir(BOUNDARY_DIR):
    print(f'\n!! BOUNDARY_DIR does not exist: {BOUNDARY_DIR}')


In [ ]:
#@title List detected campuses
from campus_svi import boundaries
found = boundaries.list_campuses()
print('Campuses found:', found or '(none)')
if not found:
    print('\nExpected files named by campus abbreviation: ui.gpkg / itb.shp / its.geojson')
    print('Shapefiles need all sidecars (.shp .shx .dbf .prj) in the same folder.')


### Credentials

Only **one** credential is needed. Add it in the Colab **Secrets** panel (key icon, left
sidebar) rather than pasting it into a cell — pasted values end up in saved notebook output.

| Secret | Needed for |
|---|---|
| `MAPILLARY_TOKEN` | Mapillary Graph API, format `MLY\|...` |

**Google needs no key.** `streetlevel` wraps Google's internal endpoints — no API key, no
billing account. The trade-off is that those endpoints are undocumented and can change without
notice, so pin the `streetlevel` version you publish results with.


In [ ]:
#@title Load the Mapillary token
import os
try:
    from google.colab import userdata
    os.environ['MAPILLARY_TOKEN'] = userdata.get('MAPILLARY_TOKEN')
except Exception as e:
    print('Secrets unavailable:', type(e).__name__)
    import getpass
    os.environ['MAPILLARY_TOKEN'] = getpass.getpass('MAPILLARY_TOKEN: ')

config.MAPILLARY_TOKEN = os.environ.get('MAPILLARY_TOKEN', '')
print('Mapillary token:', 'ok' if config.MAPILLARY_TOKEN.startswith('MLY|') else 'MISSING or malformed')


---
## 1. Build the analysis grid

A square grid is tiled over the boundary in the local UTM CRS, then filtered to cells that
meaningfully overlap the campus. Cells keep their full square shape — clipping them would
distort the bounding boxes used for Mapillary queries — but each records how much of it falls
inside, so coverage denominators stay honest.

The grid is the **analysis** unit. Google fetching uses coverage tiles instead (see section 3),
and panoramas are assigned back to cells by spatial join, so the two need not align.


In [ ]:
#@title Grid parameters
CAMPUS = 'ui'  #@param {type:'string'}
CELL_SIZE_M = 100  #@param {type:'number'}
MIN_OVERLAP = 0.05  #@param {type:'number'}

CAMPUS = CAMPUS.strip().lower()
from campus_svi import grids, google

grid = grids.build_grid(CAMPUS, cell_size_m=CELL_SIZE_M, min_overlap=MIN_OVERLAP)
path = grids.save_grid(grid, CAMPUS)
print(f'{len(grid)} cells at {CELL_SIZE_M} m')
print(f"campus area: {grid['area_inside_m2'].sum()/1e6:.2f} km2")
print('saved:', path)

tiles = google.tiles_for_campus(CAMPUS)
print(f'\nGoogle coverage tiles needed: {len(tiles)} (vs {len(grid)} grid cells)')


In [ ]:
#@title Preview the grid and the coverage tiles
import matplotlib.pyplot as plt
from campus_svi import boundaries

fig, ax = plt.subplots(figsize=(6, 6))
grid.plot(ax=ax, facecolor='none', edgecolor='#ccc', linewidth=0.25)
tiles.plot(ax=ax, facecolor='none', edgecolor='#3a6ea5', linewidth=0.8)
boundaries.load(CAMPUS).boundary.plot(ax=ax, color='#a33', linewidth=1.4)
ax.set_title(f'{CAMPUS.upper()} — {len(grid)} cells, {len(tiles)} tiles')
ax.set_axis_off()
plt.show()


---
## 2. Fetch Mapillary metadata

One bounding-box request per grid cell. Because the query is a bbox rather than a point, every
image inside the cell comes back regardless of where the road network runs.

Start with `LIMIT_CELLS = 20` as a smoke test, confirm rows are landing, then set it to `0`.


In [ ]:
#@title Fetch Mapillary (resumable)
LIMIT_CELLS = 20  #@param {type:'integer'}
SLEEP = 0.4  #@param {type:'number'}

from campus_svi import mapillary
out = mapillary.fetch_campus(CAMPUS, limit_cells=LIMIT_CELLS or None, sleep=SLEEP)

import pandas as pd
if out.exists():
    df = pd.read_csv(out)
    print(f'\n{len(df)} image records so far')
    display(df.head(3))


---
## 3. Fetch Google metadata — streetlevel, async

Two stages, both async over a shared `aiohttp` session and independently checkpointed.

**Stage A — coverage tiles.** `get_coverage_tile_async` returns *every* panorama on a zoom-17
Slippy tile (~304 m square at Indonesian latitudes). This is an area query, structurally the
same shape as a Mapillary bbox query, and it reaches panoramas a radius search never surfaces.
Tiles are coarser than the grid, so the campus needs far fewer requests than it has cells.
The tile response carries **geometry only**, and only the most recent coverage per position.

**Stage B — per-panorama enrichment.** `find_panorama_by_id_async` fills in capture date,
`source` (capture programme), `copyright_message`, `uploader`, and the `historical` list of
earlier panoramas at that position. Historical entries become their own rows — this is where
temporal depth actually comes from.

Two fields worth knowing about:

- **`is_third_party`** is derived from the pano ID string alone, so the official-versus-user-
  contributed split costs nothing and is available straight from stage A.
- **`source`** records the capture programme: `launch` is car coverage snapped to roads,
  `scout` is trekker or tripod coverage *not* snapped to roads, `innerspace` is Business View.
  On a campus with pedestrian-path trekker coverage the road-snapping caveat bites less than a
  blanket statement implies — and this field lets you measure it rather than assume it.


In [ ]:
#@title Fetch Google (resumable, async)
LIMIT_TILES = 3  #@param {type:'integer'}
ENRICH = True  #@param {type:'boolean'}
INCLUDE_HISTORICAL = True  #@param {type:'boolean'}
CONCURRENCY = 8  #@param {type:'integer'}
SLEEP_G = 0.3  #@param {type:'number'}

config.GOOGLE_CONCURRENCY = CONCURRENCY
config.GOOGLE_SLEEP = SLEEP_G
config.GOOGLE_INCLUDE_HISTORICAL = INCLUDE_HISTORICAL

out_g = google.fetch_campus(CAMPUS, enrich=ENRICH, limit_tiles=LIMIT_TILES or None)

if out_g.exists():
    dg = pd.read_csv(out_g)
    print(f'\n{len(dg)} panoramas')
    if 'is_third_party' in dg:
        print('third party :', dg['is_third_party'].astype(str).value_counts().to_dict())
    if 'capture_source' in dg:
        print('capture src :', dg['capture_source'].value_counts().to_dict())
    display(dg.head(3))


In [ ]:
#@title Check progress across all three checkpoints
from campus_svi.checkpoint import Checkpoint

for label, p in [('mapillary cells', mapillary.progress_path(CAMPUS)),
                 ('google tiles  ', google.tile_progress_path(CAMPUS)),
                 ('google panos  ', google.meta_progress_path(CAMPUS))]:
    ck = Checkpoint(p)
    print(f'{label}  {ck.summary()}')
    if ck.failed:
        print('   failed:', sorted(ck.failed)[:8], '...' if len(ck.failed) > 8 else '')

# Re-running either fetch cell retries failed units automatically.
# If requests start failing in bulk, lower CONCURRENCY before raising SLEEP.


---
## 4. Finalise — deduplicate and reclip

The gate between raw and delivery-ready. Nothing downstream reads from `data/raw/`.

**Duplicates are structural.** Coverage tiles overlap the campus edge, and a panorama's
historical list can name a pano already returned by another tile. On the Mapillary side, an
image on a bbox seam comes back from both neighbouring cell queries.

**Reclipping** matters because neither fetch unit respects the campus outline: Google tiles are
~304 m squares that overrun the boundary wholesale, and edge grid cells legitimately return
imagery from the public street outside. Points are clipped against the original boundary
polygon — not the grid or tile extent.


In [ ]:
#@title Dedup + reclip
from campus_svi import finalize
final = finalize.finalize_campus(CAMPUS)


---
## 5. Unify — per-cell table and agreement classes

Panoramas and images are assigned to grid cells by spatial join here, so the tile/cell mismatch
resolves itself.

The two sources have different native granularity: Mapillary gives many independently
timestamped images per cell, Google gives one panorama per position per capture period, dated
to the month. A raw count comparison is not apples to apples, so the cross-source metrics rest
on **binary coverage** and **temporal depth**, while density, contributor diversity and capture
programme stay source-specific.


In [ ]:
#@title Build the per-cell wide table
from campus_svi import unify
cells_path = unify.unify_campus(CAMPUS)
cells = unify.load_cells(CAMPUS)
cells.drop(columns='geometry').head()


---
## 6. Analysis and figures


In [ ]:
#@title Tables and per-campus figures
from campus_svi import analysis
analysis.run_campus(CAMPUS)
display(analysis.coverage_summary(CAMPUS).T)
display(analysis.google_profile(CAMPUS).T)


In [ ]:
#@title Agreement map
fig, ax = analysis.plot_agreement_map(CAMPUS, save=True)
plt.show()


In [ ]:
#@title Capture activity by year
analysis.plot_temporal(CAMPUS, save=True)
plt.show()


---
## 7. Repeat, then compare across campuses

Go back to **section 1**, change `CAMPUS`, and run sections 1-6 again. Everything is already
saved to Drive, so nothing is lost between sessions.


In [ ]:
#@title Cross-campus comparison
CAMPUSES = ['ui', 'itb']  #@param
CAMPUSES = [c.strip().lower() for c in CAMPUSES]

combined = unify.combine(CAMPUSES)
summary = pd.concat([analysis.coverage_summary(c) for c in CAMPUSES], ignore_index=True)
summary.to_csv(config.TABLE_DIR / 'coverage_summary_all.csv', index=False)
display(summary)

analysis.plot_coverage_bars(CAMPUSES)
plt.show()


In [ ]:
#@title Where everything lives on Drive
for k, v in config.paths().items():
    n = len(list(v.glob('*'))) if v.exists() else 0
    print(f'{k:<16} {n:>4} files   {v}')
